# HGB 보충 측정(N5) + C2 Combined

| 단계 | 내용 | 시간 |
|---|---|---|
| S | N5 재학습 → **학습곡선**(반복별 손실·정확도), **예측 시간**, **Macro Precision·Recall** | 약 45분 |
| C2 | N5 구성(픽셀+투영+HOG)을 Combined에 적용, 작성자 보류 4-fold OOF | 약 3시간 |

결과: 드라이브 `HGB_results/HGB_results_v4.csv`, 그림 `S_learning_curve.png`, `HGB_confusion_C2.png`

**실행법:** 0번 셀 실행(세션 다운 = 정상) → 1번 셀 클릭 → 런타임 → 이후 셀 실행

## 0. scikit-learn 1.8.0 설치 (세션이 재시작돼요. 정상)

In [ ]:
!pip install -q scikit-learn==1.8.0
import os; os.kill(os.getpid(), 9)

## 1. 준비

In [ ]:
!test -d /content/Machine_Learning || git clone --depth 1 https://github.com/ryu980920/Machine_Learning.git /content/Machine_Learning
from google.colab import drive
drive.mount('/content/drive')
import os, sys, time, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
OUT = '/content/drive/MyDrive/HGB_results/'; os.makedirs(OUT, exist_ok=True)
REPO = Path('/content/Machine_Learning')
sys.path.insert(0, str(REPO/'05_model_optimize/MLP/Combined/scripts'))
sys.path.insert(0, str(REPO/'05_model_optimize/MLP/Normal/scripts'))
from combined_mlp_writer_cv import load_official, load_writers
from exp006_size_com_comparison import process
import exp_cmb005_balanced_scale_stroke as L
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from skimage.feature import hog

off = load_official(REPO/'02_data/datasets/MNIST_EMNIST_official_split_112000.npz')
wr  = load_writers(REPO/'02_data/Hand_made')
WRITERS = list(L.WRITERS); LABELS = '0123456789ABCDEF'
ytr, yva = off['y_train'], off['y_val']
yhm = np.concatenate([wr[w]['y'] for w in WRITERS])
whm = np.concatenate([[w]*len(wr[w]['y']) for w in WRITERS])

Xtr_aug, tw_com, _, _ = L.prepareraw(off, wr, 'scale30_stroke28')   # N3/N5 전처리
Xva_com = process(off['X_val'], 'com', 20)
Xhm_com = np.vstack([tw_com[w] for w in WRITERS])

def shape_feats(X):
    X = X.reshape(-1, 28, 28).astype(np.float32)
    proj = np.hstack([X.sum(1), X.sum(2)]) / 255.0
    H = np.stack([hog(im, orientations=9, pixels_per_cell=(7,7), cells_per_block=(2,2)) for im in X])
    return np.hstack([proj, H]).astype(np.float32)

t = time.time()
Ftr = np.hstack([Xtr_aug, shape_feats(Xtr_aug)])
Fva = np.hstack([Xva_com, shape_feats(Xva_com)])
Fhm_map = {w: np.hstack([tw_com[w], shape_feats(tw_com[w])]) for w in WRITERS}
Fhm = np.vstack([Fhm_map[w] for w in WRITERS])
feat_s = round(time.time() - t)
print('특징 추출', feat_s, '초', Ftr.shape, Fva.shape, Fhm.shape)   # (76800,1164) (19200,1164) (2560,1164)

## 2. 공통 함수

In [ ]:
PAIRS = [('D','0'), ('0','D'), ('B','8'), ('8','B'), ('5','F'), ('7','9')]
CSV = OUT + 'HGB_results_v4.csv'
results = []

def scores(y, p, prefix):
    return {f'{prefix}_acc': accuracy_score(y, p),
            f'{prefix}_precision': precision_score(y, p, average='macro', zero_division=0),
            f'{prefix}_recall': recall_score(y, p, average='macro', zero_division=0),
            f'{prefix}_f1': f1_score(y, p, average='macro')}

def hm_detail(p):
    isd = yhm < 10
    r = dict(hm_digit=accuracy_score(yhm[isd], p[isd]), hm_letter=accuracy_score(yhm[~isd], p[~isd]))
    for w in WRITERS: r[f'w{w}'] = accuracy_score(yhm[whm==w], p[whm==w])
    for a, b in PAIRS: r[f'{a}->{b}'] = int(np.sum((yhm==LABELS.index(a)) & (p==LABELS.index(b))))
    return r

def save(row):
    results.append(row); pd.DataFrame(results).to_csv(CSV, index=False)
    print({k:(round(v,4) if isinstance(v,float) else v) for k,v in row.items()})

def hgb():
    return HistGradientBoostingClassifier(max_iter=300, early_stopping=True, validation_fraction=0.1,
                                          n_iter_no_change=10, random_state=42)

## S. N5 보충 측정 — 학습 시간·예측 시간·Precision·Recall·학습곡선

In [ ]:
m5 = hgb()
t = time.time(); m5.fit(Ftr, ytr); fit_s = time.time() - t

t = time.time(); pv = m5.predict(Fva); pred_va_s = time.time() - t
t = time.time(); ph = m5.predict(Fhm); pred_hm_s = time.time() - t

save(dict(name='S_N5_재현+보충측정', **scores(yva, pv, 'val'), **scores(yhm, ph, 'hm'), **hm_detail(ph),
          fit_s=round(fit_s), iters=m5.n_iter_, feature_s=feat_s,
          predict_val_s=round(pred_va_s, 2), predict_hm_s=round(pred_hm_s, 2),
          predict_ms_per_img=round(1000*pred_va_s/len(yva), 4)))

In [ ]:
# 학습곡선: 반복(트리 개수)마다 ① 내부 학습/검증 손실 ② 공식 Validation·손글씨 정확도
it = np.arange(len(m5.train_score_))
acc_va = [accuracy_score(yva, p) for p in m5.staged_predict(Fva)]
acc_hm = [accuracy_score(yhm, p) for p in m5.staged_predict(Fhm)]

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].plot(it, -m5.train_score_, label='Train loss (internal subsample)')
ax[0].plot(it, -m5.validation_score_, label='Validation loss (internal 10%)')
ax[0].set_xlabel('Boosting iteration (number of trees per class)'); ax[0].set_ylabel('Log loss'); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(np.arange(1, len(acc_va)+1), acc_va, label='Official Validation 19,200')
ax[1].plot(np.arange(1, len(acc_hm)+1), acc_hm, label='Hand-made 2,560 (dev)')
ax[1].set_xlabel('Boosting iteration'); ax[1].set_ylabel('Accuracy'); ax[1].legend(); ax[1].grid(alpha=.3)
plt.suptitle(f'N5 HGB learning curve (early stopped at {m5.n_iter_})'); plt.tight_layout()
plt.savefig(OUT+'S_learning_curve.png', dpi=130); plt.show()

lc = pd.DataFrame({'iteration': it, 'train_loss': -m5.train_score_, 'val_loss': -m5.validation_score_})
lc['official_val_acc'] = [np.nan] + acc_va[:len(it)-1]; lc['handmade_acc'] = [np.nan] + acc_hm[:len(it)-1]
lc.to_csv(OUT+'S_learning_curve.csv', index=False)
i_best = int(np.argmin(-m5.validation_score_))
print(f'내부 검증 손실 최소: {i_best}회 ({-m5.validation_score_[i_best]:.4f}), 마지막 {it[-1]}회 ({-m5.validation_score_[-1]:.4f})')
print(f'학습 손실 마지막: {-m5.train_score_[-1]:.4f}')

## C2. Combined — N5 구성 + 다른 작성자 3명 1,920장 추가, 작성자 보류 4-fold OOF

In [ ]:
oof = np.zeros(len(yhm), dtype=int); fold = []
for w in WRITERS:
    others = [o for o in WRITERS if o != w]
    X = np.vstack([Ftr] + [Fhm_map[o] for o in others])
    y = np.concatenate([ytr] + [wr[o]['y'] for o in others])
    m = hgb()
    t = time.time(); m.fit(X, y); fs = time.time() - t
    t = time.time(); oof[whm == w] = m.predict(Fhm_map[w]); ps = time.time() - t
    a = accuracy_score(yhm[whm==w], oof[whm==w])
    fold.append(dict(writer=w, train_n=len(y), acc=a, fit_s=round(fs), iters=m.n_iter_, predict_s=round(ps, 2)))
    print(fold[-1])
    pd.DataFrame(fold).to_csv(OUT+'C2_folds.csv', index=False)

save(dict(name='C2_Combined_N5구성', **scores(yhm, oof, 'hm'), **hm_detail(oof),
          fit_s=round(np.mean([f['fit_s'] for f in fold])), iters=round(np.mean([f['iters'] for f in fold]), 1)))

from sklearn.metrics import ConfusionMatrixDisplay
fig, ax = plt.subplots(figsize=(9, 8))
ConfusionMatrixDisplay.from_predictions(yhm, oof, labels=range(16), display_labels=list(LABELS),
                                        normalize='true', values_format='.2f', ax=ax, colorbar=False)
ax.set_title('C2 Combined (N5 features) writer-held-out OOF')
plt.tight_layout(); plt.savefig(OUT+'HGB_confusion_C2.png', dpi=130); plt.show()
pd.DataFrame(results)